<a href="https://colab.research.google.com/github/deekshitha29082006/WPT/blob/main/mini_project_dp_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import os
import time
import re
import subprocess
import urllib.request

# 1. Check whether the application is running
def app_is_running():
    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=3
        ) as response:
            return response.status == 200
    except Exception:
        return False

# 2. Start Streamlit if needed
if not app_is_running():
    print("Starting Streamlit...")

    with open("/content/streamlit.log", "a") as log:
        app_process = subprocess.Popen(
            [
                "streamlit", "run", "/content/app.py",
                "--server.port", "8501",
                "--server.address", "0.0.0.0",
                "--server.headless", "true"
            ],
            stdout=log,
            stderr=subprocess.STDOUT
        )

    for _ in range(30):
        time.sleep(2)
        if app_is_running():
            break

if not app_is_running():
    print("Streamlit failed to start. Check this log:")
    print(open("/content/streamlit.log").read()[-3000:])
else:
    print("Streamlit is working!")

    # 3. Download Cloudflare Tunnel if necessary
    binary = "/content/cloudflared"

    if not os.path.exists(binary):
        subprocess.run([
            "wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", binary
        ], check=True)

    os.chmod(binary, 0o755)

    # 4. Start a fresh tunnel and capture its output
    log_path = "/content/new_tunnel.log"

    with open(log_path, "w") as log:
        tunnel_process = subprocess.Popen(
            [
                binary, "tunnel",
                "--protocol", "http2",
                "--url", "http://127.0.0.1:8501",
                "--no-autoupdate"
            ],
            stdout=log,
            stderr=subprocess.STDOUT
        )

    print("Creating a fresh public link...")

    new_url = None

    for _ in range(45):
        time.sleep(2)

        if tunnel_process.poll() is not None:
            break

        with open(log_path, "r") as log:
            output = log.read()

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            output
        )

        if match:
            new_url = match.group(0)
            break

    if new_url:
        print("\nYOUR NEW APPLICATION LINK:")
        print(new_url)
        print("\nOpen this exact URL in your browser.")
    else:
        print("No public URL was generated. Tunnel output:")
        print(open(log_path).read()[-4000:])

Streamlit is working!
Creating a fresh public link...

YOUR NEW APPLICATION LINK:
https://everywhere-duties-arcade-expressed.trycloudflare.com

Open this exact URL in your browser.
